# Agentic RAG with AWS

## 📖 Overview

This notebook demonstrates **Agentic RAG** using AWS services:
- **Amazon Bedrock Claude** with tool use for autonomous reasoning
- **AWS OpenSearch Serverless** as one of many tools
- **Custom tools** for extended capabilities

### What is Agentic RAG?

Agentic RAG gives the LLM **autonomy to decide**:
- **When** to retrieve information
- **What** to retrieve (which queries to run)
- **Which tools** to use beyond just retrieval
- **How many** retrieval rounds needed
- **What actions** to take based on findings

**Traditional RAG (Passive):**
```
User: "What's the weather in Seattle?"
→ System: ALWAYS retrieves from knowledge base
→ Returns outdated or irrelevant info
```

**Agentic RAG (Active):**
```
User: "What's the weather in Seattle?"
→ Agent thinks: "This needs current data, not knowledge base"
→ Agent chooses: weather_api tool instead of retrieve tool
→ Returns current weather
```

### Why Agentic?

**Problems with Static RAG:**
- Always retrieves, even when not needed
- Can't access external data sources
- No ability to perform calculations
- Can't chain multiple operations
- Fixed pipeline, no flexibility

**Agentic RAG Benefits:**
- **Intelligent**: Decides when retrieval helps
- **Flexible**: Can use multiple tools
- **Autonomous**: Self-directed reasoning
- **Extensible**: Easy to add new tools
- **Powerful**: Multi-step reasoning

### When to Use

✅ **Good for:**
- Complex multi-step tasks
- Need for external data/APIs
- Calculations or data processing
- Uncertain retrieval needs
- Research and analysis tasks
- Interactive applications

❌ **Not ideal for:**
- Simple factual Q&A
- When retrieval always needed
- Tight latency requirements
- Very tight budgets (more LLM calls)
- Need deterministic behavior

### Architecture

```mermaid
graph TB
    A[User Query] --> B[Agent Reasoning<br/>Claude with Tools]
    
    B --> C{What tool to use?}
    
    C -->|Need docs| D1[retrieve_documents tool]
    C -->|Need calc| D2[calculator tool]
    C -->|Need data| D3[external_api tool]
    C -->|Search web| D4[web_search tool]
    C -->|Have answer| D5[No tool, respond]
    
    D1 --> E1[OpenSearch Query]
    D2 --> E2[Python Eval]
    D3 --> E3[API Call]
    D4 --> E4[Web Search]
    
    E1 --> F[Tool Results]
    E2 --> F
    E3 --> F
    E4 --> F
    
    F --> G[Agent Analyzes Results]
    G --> H{Need more tools?}
    
    H -->|Yes| C
    H -->|No| I[Generate Final Answer]
    I --> J[Return to User]

    style A fill:#e1f5ff
    style B fill:#fff3e0
    style C fill:#f3e5f5
    style D1 fill:#e8f5e9
    style D2 fill:#e8f5e9
    style D3 fill:#e8f5e9
    style D4 fill:#e8f5e9
    style G fill:#ffe0b2
    style I fill:#c8e6c9
    style J fill:#b3e5fc
```

## 1️⃣ Setup & Imports

In [ ]:
import sys
import json
from typing import List, Dict, Callable, Any, Optional
import time
import re

sys.path.append('..')

from aws_utils.opensearch_manager import OpenSearchManager
from aws_utils.bedrock_client import BedrockEmbeddings, BedrockLLM
from aws_utils.rag_evaluator import RAGEvaluator

print("✓ Imports successful")

# Expected output:
# ✓ Imports successful

## 2️⃣ Configuration

In [ ]:
# AWS Configuration
AWS_REGION = 'us-west-2'
INDEX_NAME = 'agentic_rag_docs'

# Model Configuration
EMBEDDING_MODEL = 'amazon.titan-embed-text-v2:0'
AGENT_MODEL = 'us.anthropic.claude-opus-4-1-20250805-v1:0'  # Opus for reasoning

# Agent Parameters
MAX_ITERATIONS = 5  # Maximum tool use rounds
RETRIEVAL_TOP_K = 5  # Documents per retrieval

print(f"Configuration:")
print(f"  Agent model: {AGENT_MODEL.split('.')[-1][:40]}")
print(f"  Max iterations: {MAX_ITERATIONS}")
print(f"  Retrieval per call: Top-{RETRIEVAL_TOP_K}")

# Expected output:
# Configuration:
#   Agent model: claude-opus-4-1-20250805-v1:0
#   Max iterations: 5
#   Retrieval per call: Top-5

## 3️⃣ Initialize Services

In [ ]:
opensearch = OpenSearchManager(region=AWS_REGION, index_name=INDEX_NAME)
embedder = BedrockEmbeddings(AWS_REGION, EMBEDDING_MODEL)
agent_llm = BedrockLLM(AWS_REGION, AGENT_MODEL, temperature=0.7)

print("✓ Services initialized")

# Expected output:
# ✓ Services initialized

## 4️⃣ Load Sample Knowledge Base

In [ ]:
sample_documents = [
    # AWS Bedrock information
    "AWS Bedrock provides access to Claude Opus ($15/$75 per million tokens), Sonnet ($3/$15), and Haiku ($0.25/$1.25).",
    "Bedrock supports batch inference which is 50% cheaper than real-time for async workloads.",
    "Claude Opus excels at complex analysis and multi-step reasoning with up to 200K context window.",
    
    # RAG information
    "RAG systems combine OpenSearch for retrieval with Bedrock for generation, typically costing $0.05-0.10 per query.",
    "Production RAG requires monitoring precision, recall, latency, and costs through CloudWatch metrics.",
    "Agentic RAG gives LLMs autonomy to decide when to retrieve, what tools to use, and how many steps to take.",
    
    # Company information
    "Anthropic was founded in 2021 by former OpenAI researchers including Dario and Daniela Amodei.",
    "Claude uses Constitutional AI training to ensure helpful, harmless, and honest responses.",
    
    # Technical details
    "Vector embeddings with Titan V2 are 1024-dimensional using cosine similarity for search.",
    "HNSW algorithm in OpenSearch enables sub-100ms vector search on millions of documents.",
    "Tool use in Claude allows function calling for calculators, APIs, databases, and custom actions.",
    
    # Best practices
    "Cache embeddings in ElastiCache to reduce Bedrock API calls by 80% for repeated queries.",
    "Use Haiku for simple tasks and Opus for complex reasoning to optimize cost-quality trade-offs.",
    "Implement circuit breakers and retry logic with exponential backoff for production reliability.",
]

# Index documents
opensearch.create_index(
    embedding_dim=embedder.get_embedding_dimension(),
    force_recreate=True
)

print("Indexing knowledge base...")
documents = []
for i, text in enumerate(sample_documents):
    embedding = embedder.embed_text(text)
    documents.append({
        'text': text,
        'embedding': embedding,
        'metadata': {'doc_id': i}
    })

opensearch.index_documents(documents)
print(f"✓ Indexed {len(documents)} documents")

# Expected output:
# Indexing knowledge base...
# ✓ Indexed 15 documents

## 5️⃣ Define Agent Tools

Tools the agent can autonomously choose to use.

In [ ]:
class Tool:
    """Base class for agent tools"""
    def __init__(self, name: str, description: str, func: Callable):
        self.name = name
        self.description = description
        self.func = func
    
    def execute(self, **kwargs) -> str:
        try:
            result = self.func(**kwargs)
            return str(result)
        except Exception as e:
            return f"Error executing {self.name}: {str(e)}"

# Tool 1: Document Retrieval
def retrieve_documents(query: str, top_k: int = 5) -> str:
    """Retrieve relevant documents from knowledge base"""
    query_emb = embedder.embed_text(query)
    results = opensearch.vector_search(query_emb, top_k=top_k)
    
    if not results:
        return "No relevant documents found."
    
    output = "Retrieved Documents:\n\n"
    for i, doc in enumerate(results, 1):
        output += f"{i}. {doc['text']}\n\n"
    
    return output

# Tool 2: Calculator
def calculator(expression: str) -> str:
    """Evaluate mathematical expressions"""
    try:
        # Safe eval with limited scope
        allowed_names = {"__builtins__": {}}
        result = eval(expression, allowed_names)
        return f"Result: {result}"
    except Exception as e:
        return f"Calculation error: {str(e)}"

# Tool 3: Date/Time
def get_current_time() -> str:
    """Get current date and time"""
    from datetime import datetime
    now = datetime.now()
    return f"Current date and time: {now.strftime('%Y-%m-%d %H:%M:%S')}"

# Tool 4: String Operations
def string_operations(operation: str, text: str) -> str:
    """Perform string operations (upper, lower, reverse, length)"""
    operations = {
        'upper': text.upper(),
        'lower': text.lower(),
        'reverse': text[::-1],
        'length': str(len(text)),
        'words': str(len(text.split()))
    }
    
    if operation in operations:
        return operations[operation]
    else:
        return f"Unknown operation. Available: {', '.join(operations.keys())}"

# Register tools
available_tools = [
    Tool(
        name="retrieve_documents",
        description="Search the knowledge base for relevant documents. Use when you need factual information about AWS, Bedrock, RAG, or technical topics.",
        func=retrieve_documents
    ),
    Tool(
        name="calculator",
        description="Evaluate mathematical expressions. Use for calculations, cost estimates, or any math operations.",
        func=calculator
    ),
    Tool(
        name="get_current_time",
        description="Get the current date and time. Use when asked about current time or date.",
        func=get_current_time
    ),
    Tool(
        name="string_operations",
        description="Perform string operations (upper, lower, reverse, length, words). Use for text manipulation tasks.",
        func=string_operations
    ),
]

print(f"Registered {len(available_tools)} tools:")
for tool in available_tools:
    print(f"  - {tool.name}: {tool.description[:60]}...")

# Expected output:
# Registered 4 tools:
#   - retrieve_documents: Search the knowledge base for relevant documents. Use when...
#   - calculator: Evaluate mathematical expressions. Use for calculations, co...
#   - get_current_time: Get the current date and time. Use when asked about curr...
#   - string_operations: Perform string operations (upper, lower, reverse, length,...

## 6️⃣ Agent Reasoning Loop

In [ ]:
def agent_reasoning_loop(query: str, 
                        tools: List[Tool],
                        max_iterations: int = 5) -> Dict:
    """
    Agent reasoning with autonomous tool use
    
    Returns:
        Dict with answer, steps, metadata
    """
    start_time = time.time()
    conversation_history = []
    steps = []
    
    print(f"Agent Query: {query}\n")
    print("="*70 + "\n")
    
    # Build tool descriptions for agent
    tools_desc = "\n".join([
        f"- {tool.name}: {tool.description}"
        for tool in tools
    ])
    
    # System prompt
    system_prompt = f"""
You are an autonomous agent that can use tools to answer questions.

Available Tools:
{tools_desc}

For each step:
1. Analyze what information you need
2. Decide if you need to use a tool
3. If yes, respond with: TOOL: tool_name | ARGS: {{"arg1": "value1"}}
4. If no, respond with: ANSWER: your final answer

Be strategic: only use tools when necessary. Think step-by-step.
"""
    
    current_query = f"User Question: {query}\n\nWhat is your next step?"
    
    for iteration in range(1, max_iterations + 1):
        print(f"Iteration {iteration}/{max_iterations}")
        
        # Agent thinks
        print("  Agent thinking...")
        response = agent_llm.generate(
            current_query,
            system_prompt=system_prompt,
            temperature=0.7
        )
        
        print(f"  Response: {response[:100]}...\n")
        
        # Check if agent wants to use a tool
        if "TOOL:" in response:
            # Parse tool use
            tool_match = re.search(r'TOOL:\s*(\w+)', response)
            args_match = re.search(r'ARGS:\s*({.*?})', response, re.DOTALL)
            
            if tool_match:
                tool_name = tool_match.group(1)
                
                # Parse args
                tool_args = {}
                if args_match:
                    try:
                        tool_args = json.loads(args_match.group(1))
                    except:
                        # Fallback: extract key-value pairs
                        args_text = args_match.group(1)
                        # Simple parsing for query="..." patterns
                        query_match = re.search(r'"query"\s*:\s*"([^"]+)"', args_text)
                        if query_match:
                            tool_args['query'] = query_match.group(1)
                
                # Find and execute tool
                tool = next((t for t in tools if t.name == tool_name), None)
                
                if tool:
                    print(f"  🔧 Using tool: {tool_name}")
                    print(f"     Args: {tool_args}")
                    
                    tool_result = tool.execute(**tool_args)
                    
                    print(f"     Result: {tool_result[:100]}...\n")
                    
                    steps.append({
                        'iteration': iteration,
                        'action': f"Used {tool_name}",
                        'args': tool_args,
                        'result': tool_result[:200]
                    })
                    
                    # Update conversation
                    current_query = f"Tool Result from {tool_name}:\n{tool_result}\n\nWhat is your next step?"
                else:
                    print(f"  ✗ Tool not found: {tool_name}\n")
                    break
        
        elif "ANSWER:" in response:
            # Agent has final answer
            answer = response.split("ANSWER:", 1)[1].strip()
            print(f"  ✓ Agent has final answer\n")
            
            steps.append({
                'iteration': iteration,
                'action': 'Generated final answer',
                'result': answer[:200]
            })
            
            total_time = time.time() - start_time
            
            return {
                'answer': answer,
                'steps': steps,
                'iterations': iteration,
                'completed': True,
                'metadata': {
                    'total_time': total_time,
                    'tools_used': len([s for s in steps if 'Used' in s['action']])
                }
            }
        else:
            # Agent response unclear
            print(f"  ? Unclear response, treating as answer\n")
            
            total_time = time.time() - start_time
            
            return {
                'answer': response,
                'steps': steps,
                'iterations': iteration,
                'completed': True,
                'metadata': {
                    'total_time': total_time,
                    'tools_used': len([s for s in steps if 'Used' in s['action']])
                }
            }
    
    # Max iterations reached
    total_time = time.time() - start_time
    
    return {
        'answer': "Maximum iterations reached without final answer.",
        'steps': steps,
        'iterations': max_iterations,
        'completed': False,
        'metadata': {
            'total_time': total_time,
            'tools_used': len([s for s in steps if 'Used' in s['action']])
        }
    }

print("✓ Agent reasoning loop ready")

# Expected output:
# ✓ Agent reasoning loop ready

## 7️⃣ Test Agentic RAG

In [ ]:
# Test queries that require different tool combinations
test_queries = [
    "What does AWS Bedrock cost for Claude Opus?",  # Needs retrieval
    "If I run 10,000 queries at $0.08 each, what's the total cost?",  # Needs calculator
    "What time is it?",  # Needs time tool
    "How many words are in 'AWS Bedrock provides foundation models'?",  # Needs string tool
]

for i, query in enumerate(test_queries, 1):
    print(f"\n{'#'*70}")
    print(f"# Test {i}")
    print(f"{'#'*70}\n")
    
    result = agent_reasoning_loop(query, available_tools, MAX_ITERATIONS)
    
    print("="*70)
    print("RESULTS")
    print("="*70)
    
    print(f"\n📊 Summary:")
    print(f"  Iterations: {result['iterations']}/{MAX_ITERATIONS}")
    print(f"  Tools used: {result['metadata']['tools_used']}")
    print(f"  Completed: {result['completed']}")
    print(f"  Time: {result['metadata']['total_time']:.2f}s")
    
    print(f"\n🔄 Agent Steps:")
    for step in result['steps']:
        print(f"  {step['iteration']}. {step['action']}")
        if 'args' in step:
            print(f"      Args: {step['args']}")
    
    print(f"\n✅ Final Answer:\n{result['answer'][:300]}...")
    print("\n" + "#"*70 + "\n")

# Expected output:
# [Each query showing autonomous tool selection and multi-step reasoning]

## 8️⃣ Complex Multi-Tool Task

In [ ]:
# Complex task requiring multiple tools
complex_query = """
I want to build a RAG system that handles 100,000 queries per month.
1. Find the cost information for Bedrock
2. Calculate the monthly cost
3. Tell me if this is affordable for a startup
"""

print("Complex Multi-Step Task\n" + "="*70 + "\n")

result = agent_reasoning_loop(complex_query, available_tools, MAX_ITERATIONS)

print("\n" + "="*70)
print("ANALYSIS")
print("="*70)

print(f"\n📊 Execution Trace:")
for i, step in enumerate(result['steps'], 1):
    print(f"\nStep {i}: {step['action']}")
    if 'args' in step:
        print(f"  Arguments: {json.dumps(step['args'], indent=4)}")
    if 'result' in step:
        print(f"  Result: {step['result'][:150]}...")

print(f"\n💡 Agent Demonstrated:")
print(f"  ✓ Autonomous decision-making")
print(f"  ✓ Multi-tool coordination")
print(f"  ✓ Step-by-step reasoning")
print(f"  ✓ Information synthesis")

print(f"\n✅ Final Answer:\n{result['answer']}")

# Expected output:
# [Trace showing agent retrieving cost info, calculating, then providing assessment]

## 9️⃣ Comparison: Agentic vs Traditional RAG

In [ ]:
comparison_query = "What time is it and how much does AWS Bedrock cost?"

print("="*70)
print("AGENTIC RAG")
print("="*70 + "\n")

agentic_result = agent_reasoning_loop(comparison_query, available_tools, MAX_ITERATIONS)

print("\n" + "="*70)
print("TRADITIONAL RAG (Always Retrieves)")
print("="*70 + "\n")

# Traditional: always retrieves
traditional_start = time.time()
query_emb = embedder.embed_text(comparison_query)
trad_results = opensearch.vector_search(query_emb, top_k=3)
trad_answer = agent_llm.generate_with_context(
    comparison_query,
    [r['text'] for r in trad_results]
)
traditional_time = time.time() - traditional_start
print(f"✓ Retrieved and generated in {traditional_time:.2f}s\n")

# Comparison
print("="*70)
print("COMPARISON")
print("="*70)

print(f"\n🎯 Intelligence:")
print(f"  Agentic: Used {agentic_result['metadata']['tools_used']} tools strategically")
print(f"  Traditional: Always retrieves (may not have time info)")

print(f"\n⏱️  Time:")
print(f"  Agentic: {agentic_result['metadata']['total_time']:.2f}s ({agentic_result['iterations']} iterations)")
print(f"  Traditional: {traditional_time:.2f}s (1 retrieval)")

print(f"\n📝 Answers:\n")
print(f"Agentic:\n{agentic_result['answer'][:300]}...\n")
print(f"Traditional:\n{trad_answer[:300]}...")

print(f"\n💡 Analysis:")
print(f"  - Agentic can use multiple tools (time + retrieval)")
print(f"  - Traditional limited to knowledge base only")
print(f"  - Agentic adapts to query requirements")
print(f"  - Traditional follows fixed pipeline")

# Expected output:
# [Showing agentic using multiple tools vs traditional limited to retrieval]

## 🔟 Summary & Key Takeaways

### What We Built

✅ Autonomous agent with tool use
✅ Multiple tools (retrieval, calculator, time, string ops)
✅ Reasoning loop with decision-making
✅ Multi-step task execution
✅ Tool coordination and synthesis

### Performance Characteristics

- **Latency**: Variable (1-5+ iterations)
- **Cost**: Higher (multiple LLM reasoning calls)
- **Flexibility**: Can solve diverse problems
- **Intelligence**: Autonomous decision-making

### When to Use Agentic RAG

**Use Agentic when:**
- Tasks require multiple steps
- Need external tools/APIs
- Uncertain what's needed upfront
- Complex reasoning required
- Want flexible behavior
- Interactive applications

**Skip Agentic when:**
- Simple Q&A
- Always need retrieval
- Latency critical
- Very tight budget
- Need deterministic behavior
- Security concerns with tool access

### Key Insights

1. **Autonomy**: Agent decides what tools to use
2. **Flexibility**: Can handle diverse task types
3. **Intelligence**: Step-by-step reasoning
4. **Extensible**: Easy to add new tools
5. **Powerful**: Multi-tool coordination

### Best Practices

1. **Clear Tool Descriptions**: Help agent choose correctly
2. **Safe Tool Execution**: Sandbox dangerous operations
3. **Max Iterations**: Prevent infinite loops
4. **Tool Results**: Return clear, parseable output
5. **Error Handling**: Graceful tool failures

### Advanced Topics

- **Tool Chains**: Pre-defined tool sequences
- **Hierarchical Agents**: Agents that spawn sub-agents
- **Memory**: Agents with long-term memory
- **Multi-agent**: Multiple agents collaborating
- **Learned Tool Use**: Fine-tune tool selection

### Limitations

1. **Unpredictable**: Agent may choose unexpected tools
2. **Higher Cost**: Multiple reasoning rounds
3. **Longer Latency**: Sequential tool use
4. **Tool Errors**: Cascading failures
5. **Security**: Need to sandbox tool execution

### Real-world Tools

Production agentic RAG might include:
- **Database queries**: SQL tool
- **API calls**: Weather, stock prices, etc.
- **File operations**: Read, write, search
- **Web search**: Current information
- **Code execution**: Python, JavaScript
- **Email/Slack**: Communication tools

### Next Steps

- **More Tools**: Expand agent capabilities
- **Tool Learning**: Train which tools to use
- **Parallel Tools**: Execute multiple simultaneously
- **Agent Memory**: Remember past interactions
- **Multi-agent Systems**: Coordinate multiple agents

---

## 🎉 Phase 2 Progress!

**Progress**: 12/37 patterns (32%)
- Phase 1: 10/10 ✅ Complete
- Phase 2: 2/12 ✅ Multi-modal + Agentic RAG

**Next**: Corrective RAG (CRAG) - Self-correction mechanisms

## 🧹 Cleanup

In [ ]:
# Uncomment to delete index
# opensearch.delete_index(INDEX_NAME)
# print(f"✓ Deleted index: {INDEX_NAME}")

print("\nTo delete the index later:")
print(f"  opensearch.delete_index('{INDEX_NAME}')")

# Expected output:
# 
# To delete the index later:
#   opensearch.delete_index('agentic_rag_docs')